# Fine-tune YOLOv8n on CCT20 subset (Colab GPU)

Clones the repo, rebuilds the small data subset from scratch (data is never committed to git), fine-tunes YOLOv8n, and evaluates per-class precision/recall/mAP on the unseen-location test set.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
!git clone https://github.com/ada-th25/camera-trap-site-robustness.git
%cd camera-trap-site-robustness

Install only the lightweight packages individually. We deliberately do **not** run `pip install -r requirements.txt` here — that file was frozen from a local Mac (CPU/MPS) environment and includes a `torch` build that would overwrite Colab's preinstalled CUDA-matched PyTorch, silently losing GPU acceleration.

In [ ]:
!pip install -q ultralytics pandas pyyaml requests tqdm

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Rebuild the data pipeline (same scripts as local dev)

In [ ]:
!mkdir -p data/raw/annotations/extracted && \
curl -L -o data/raw/annotations/eccv_18_annotations.tar.gz https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz && \
tar -xzf data/raw/annotations/eccv_18_annotations.tar.gz -C data/raw/annotations/extracted

from pathlib import Path

expected = Path('data/raw/annotations/extracted/eccv_18_annotation_files/train_annotations.json')
assert expected.exists(), f'{expected} missing - annotation download/extract failed, check the cell output above'
print('annotations ready')

In [ ]:
!PYTHONPATH=. python scripts/download_subset.py

In [ ]:
!PYTHONPATH=. python scripts/build_manifest.py
!PYTHONPATH=. python scripts/convert_to_yolo.py

from pathlib import Path

assert Path('data/yolo/dataset.yaml').exists(), 'dataset.yaml missing - check the cell output above for errors'
print('dataset.yaml ready')

## Fine-tune

`patience=20`: stop early if val mAP hasn't improved in 20 epochs, so we don't burn free-tier GPU time on a plateaued run.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='data/yolo/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project='runs/finetune',
    name='cct20_yolov8n',
)

## Evaluate on the unseen-location test set (location 125)

This gives real per-class precision/recall/mAP50/mAP50-95, computed by Ultralytics' built-in validator — the same metric machinery, now with matching classes.

In [ ]:
import glob

# Ultralytics nests under its own default runs/detect/ regardless of our
# project= setting, so locate best.pt rather than hardcoding the path.
best_pt_matches = glob.glob('runs/**/best.pt', recursive=True)
assert len(best_pt_matches) == 1, f'expected exactly one best.pt, found {best_pt_matches}'
best_pt_path = best_pt_matches[0]
print('using weights:', best_pt_path)

best = YOLO(best_pt_path)
metrics = best.val(data='data/yolo/dataset.yaml', split='test')
print(metrics.box.map50, metrics.box.map)

## Download the fine-tuned weights back to your machine

In [ ]:
from google.colab import files

files.download(best_pt_path)

## Step 5: GPU benchmarking (PyTorch vs. ONNX)

Installing `onnxruntime-gpu` *before* exporting matters: Ultralytics'
export step auto-installs a plain `onnxruntime` (CPU-only) if nothing is
already present, and that package conflicts with `onnxruntime-gpu` - whichever
gets imported first silently wins. Pre-installing the GPU variant here means
the auto-installer sees it already satisfied and leaves it alone, so ONNX
inference actually runs on the GPU instead of silently falling back to CPU.

In [ ]:
!pip install -q onnxruntime-gpu onnx onnxslim

onnx_path = best.export(format='onnx', imgsz=640)
print('exported to:', onnx_path)

In [ ]:
import os

# batch=1 for true single-image latency, matching how infer.py is actually used
# (one image at a time), not batched throughput.

print('=== PyTorch, GPU ===')
pt_metrics = YOLO(best_pt_path).val(data='data/yolo/dataset.yaml', split='test', device=0, batch=1, verbose=False)
print('speed:', pt_metrics.speed)

print()
print('=== ONNX, GPU ===')
onnx_metrics = YOLO(onnx_path).val(data='data/yolo/dataset.yaml', split='test', device=0, batch=1, verbose=False)
print('speed:', onnx_metrics.speed)

print()
print('file sizes (MB):')
print('  .pt:  ', round(os.path.getsize(best_pt_path) / 1e6, 2))
print('  .onnx:', round(os.path.getsize(onnx_path) / 1e6, 2))